# **RandomForestClassifier**

---

Versão 2.0.0

## **Importação das bibliotecas**

In [45]:
import pandas as pd
import numpy as np
import binascii
import seaborn as sns
from micromlgen import port
import matplotlib.pyplot as plt
from datetime import datetime

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## **Leitura do dataset e Separação de features e target**

In [46]:
# 1. Leitura dos dados

# Caminho do CSV
#arquivo = "../data/dataset/UFDS181125EE05.csv" # Escala Macro 05A
# arquivo = "../data/dataset/UFDS270426E05.csv" # Escala Macro 05A
#arquivo = "../data/dataset/UFDS170326EE10.csv" # Escala Macro 10A
arquivo = "../data/dataset/UFDS080526EE10.csv" # Recente

# Lê o CSV
dados = pd.read_csv(arquivo)

# Tratamento de valores ausentes
dados = dados.fillna(-100)

# Separação de features e target
X = dados.iloc[:, 2:]
y = dados['ambiente']

In [ ]:
dados

## **Codificação das classes (LabelEncoder)**

In [47]:
# 2. Codificação das classes

# Conversão das classes para números
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Tabela do mapeamento realizado
df_labels = pd.DataFrame({'Ambiente': le.classes_, 'LabelEncoder': range(len(le.classes_))})
print("\nMapeamento das Classes:")
print(df_labels)


Mapeamento das Classes:
                               Ambiente  LabelEncoder
0            Auditório de Eng. Elétrica             0
1          Coordenação de Eng. Elétrica             1
2   Coordenação de pós em Eng. Elétrica             2
3                        Lab. Analítico             3
4                   Lab. Informática 09             4
5                      Lab. de Ciências             5
6      Lab. de Instrumentação biomédica             6
7          Lab. de Medição e Calibração             7
8              Lab. de Telecomunicações             8
9                Lab. de fibras ópticas             9
10           Lab. de redes convergentes            10
11                      Lab. química 01            11
12                      Lab. química 03            12
13     Núcleo de Pesquisa e Atendimento            13
14                    PET Eng. Elétrica            14
15            Pesquisa em Eng. Elétrica            15
16                   Práticas didáticas            16
17 

## **Modelo utilizado**

In [54]:
# 3. Modelo a ser treinado

rf_model = RandomForestClassifier(
    n_estimators=15,          # Número de árvores
    max_depth=None,              # Profundidade máxima
    max_features='sqrt',       # Subconjunto aleatório de features
    random_state=42
)

In [ ]:
# ===== HIPERPARÂMETROS =====
N_estimators = [50, 100, 150]
Max_depth = [5, 10, 15, None]

resultados = []

# ===== LOOP =====
for n_est in N_estimators:
    for depth in Max_depth:

        medias = []

        for i in range(20):

            rf_model = RandomForestClassifier(
                n_estimators=n_est,
                max_depth=depth,
                max_features='sqrt'
                random_state=42
            )

            kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=i)

            cv_scores = cross_val_score(
                rf_model,
                X,
                y_encoded,
                cv=kfold,
                scoring='accuracy',
                n_jobs=-1
            )

            media_fold = cv_scores.mean()
            medias.append(media_fold)

            print(f"[n_estimators={n_est}, max_depth={depth}] Rodada {i+1} - Acurácia: {media_fold:.4f}")

        # ===== RESULTADO FINAL =====
        media_final = np.mean(medias)
        desvio_final = np.std(medias)

        print("\n======================================")
        print(f"n_estimators={n_est} | max_depth={depth}")
        print(f"Acurácia média: {media_final:.4f}")
        print(f"Desvio padrão: {desvio_final:.4f}")
        print("======================================\n")

        resultados.append({
            "n_estimators": n_est,
            "max_depth": depth,
            "media_acuracia": media_final,
            "desvio_padrao": desvio_final
        })

# ===== DATAFRAME FINAL =====
df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.sort_values(by="media_acuracia", ascending=False)

df_resultados.to_csv("../data/hiperParametros/resultados_randomforest10.csv", index=False)

print("\nTop resultados:")
print(df_resultados.head(10))

## **Validação Cruzada - KFold**

In [55]:
# 4. Validação Cruzada

medias = []

for i in range(10):

    kfold = StratifiedKFold(n_splits=5, shuffle=True)

    cv_scores = cross_val_score(
        rf_model,
        X,
        y_encoded,
        cv=kfold,
        scoring='accuracy'
    )

    media_fold = cv_scores.mean()
    medias.append(media_fold)

    print(f"Rodada {i+1} - Acurácia média: {media_fold:.4f}")

# Resultado final consolidado
media_final = np.mean(medias)
desvio_final = np.std(medias)

print("\n======================================")
print("=========== Random Forest ============")
print("======================================")
print("Acurácia média (10 execuções): {:.4f}".format(media_final))
print("Desvio padrão entre execuções: {:.4f}".format(desvio_final))

Rodada 1 - Acurácia média: 1.0000
Rodada 2 - Acurácia média: 0.9944
Rodada 3 - Acurácia média: 1.0000
Rodada 4 - Acurácia média: 0.9889
Rodada 5 - Acurácia média: 1.0000
Rodada 6 - Acurácia média: 1.0000
Rodada 7 - Acurácia média: 0.9889
Rodada 8 - Acurácia média: 0.9889
Rodada 9 - Acurácia média: 0.9944
Rodada 10 - Acurácia média: 1.0000

=========== Random Forest ============
Acurácia média (10 execuções): 0.9956
Desvio padrão entre execuções: 0.0048


## **Separação dos dados**

In [ ]:
# 5. Divisão treino/teste stratify=y_encoded, 

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded,test_size=0.2, stratify=y_encoded, random_state=12)

## **Treinamento do modelo**

In [ ]:
# 6. Treinamento Final

rf_model.fit(X_train, y_train)

preds = rf_model.predict(X_test)

acc = accuracy_score(y_test, preds)

print(f"\nAcurácia no conjunto de teste: {acc:.4f} ({acc*100:.2f}%)")
print("\nAcurácia no conjunto de teste:", accuracy_score(y_test, preds))

## **Detector de melhor Modelo**

In [ ]:
max_tentativas = 100
tentativa = 0
acc = 0
minimo = 1

while acc < minimo and tentativa < max_tentativas:
    tentativa += 1
    # 5. Divisão treino/teste

    X_train, X_test, y_train, y_test = train_test_split(X, y_encoded,test_size=0.2, stratify=y_encoded, random_state=tentativa)

    # 6. Treinamento Final

    rf_model.fit(X_train, y_train)

    preds = rf_model.predict(X_test)

    acc = accuracy_score(y_test, preds)

    print(f"\nAcurácia no conjunto de teste: {acc:.4f} ({acc*100:.2f}%)")
    print("\nAcurácia no conjunto de teste:", accuracy_score(y_test, preds))
    print("Tentativa: ", tentativa)
if acc >= minimo:
    print(f"\nAtingiu {acc*100:.2f}% de acurácia!")
else:
    print(f"\nNão atingiu {minimo:.2f}% dentro do limite.")

## **Matriz de Confusão**

In [ ]:
# Matriz de Confusão em um único DataFrame

cm = confusion_matrix(y_test, preds)

cm_df = pd.DataFrame(
    cm,
    index=le.classes_,
    columns=le.classes_
)

# Transformando em um único DataFrame estruturado
cm_df = cm_df.reset_index()
cm_df = cm_df.rename(columns={'index': 'Classe_Real'})

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_)

plt.xlabel("Classe Predita")
plt.ylabel("Classe Real")

plt.savefig("../images/UmatrizConfusaoRandomForestVirtual10.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
print("\nMatriz de Confusão (DataFrame único):")
cm_df

In [ ]:
print("\nRelatório de Classificação:")
print(classification_report(y_test, preds))

In [ ]:
print("\nMatriz de Confusão:")
print(confusion_matrix(y_test, preds))

## **Conversão para C**

In [ ]:
# Mapeamento de classes (índice → nome da classe)

class_map = {i: classe for i, classe in enumerate(le.classes_)}

In [ ]:
with open('../firmware/models/RandomForestClassifierMacro10U.h', 'w') as file:
    file.write(port(rf_model, classmap=class_map))

In [ ]:
class_map

## **Conversão de MAC para CRC32**

In [ ]:
# Extrai apenas as colunas que são MACs
macs = list(dados.columns[2:])

def mac_to_crc32(mac):
    return binascii.crc32(mac.encode()) & 0xFFFFFFFF


print("uint32_t macHashList[] = {")

for i, mac in enumerate(macs):
    crc = mac_to_crc32(mac)

    # Imprime o valor
    print(f"0x{crc:08X}, ", end="")

    # Quebra linha a cada 5 elementos
    if (i + 1) % 5 == 0:
        print()

# Garante quebra final se não for múltiplo de 5
if len(macs) % 5 != 0:
    print()

print("};")

In [ ]:
#Bytes Reais

# Extrai apenas as colunas que são MACs
macs = list(dados.columns[2:])

def mac_to_crc32(mac):
    # Remove separadores ":" e converte para bytes reais
    mac_bytes = bytes.fromhex(mac.replace(":", "").replace("-", ""))
    return binascii.crc32(mac_bytes) & 0xFFFFFFFF

print("uint32_t macHashList[] = {")

for i, mac in enumerate(macs):
    crc = mac_to_crc32(mac)

    print(f"0x{crc:08X}, ", end="")

    # Quebra linha a cada 5 elementos
    if (i + 1) % 5 == 0:
        print()

if len(macs) % 5 != 0:
    print()

print("};")

In [ ]:
macs

# **Simulação de Teste em Campo**

## **Leitura do dataset e Separação de features e target**

In [ ]:
# 1. Leitura dos dados

#arquivo2 = "../data/dataset/UFDS170326EE10.csv"
arquivo2 = "../data/dataset/UFDS080526EE10.csv"

# Lê o CSV
dados2 = pd.read_csv(arquivo2)

# Tratamento de valores ausentes
dados2 = dados2.fillna(-100)

# Tratamento das Colunas
dados2.columns = dados2.columns.str.lower().str.strip()

# Guardar colunas importantes
col_num = dados2["num"]
col_ambiente = dados2["ambiente"]

# Features (remove num e local)
X_novo = dados2.iloc[:, 2:]

cols_dados = list(dados.columns)
cols_dados2 = list(dados2.columns)

In [ ]:
set_dados = set(cols_dados)
set_dados2 = set(cols_dados2)

In [ ]:
# Colunas Extras

colunas_extras = list(set_dados2 - set_dados)

if colunas_extras:
    print("Colunas removidas de dados2:")
    print(colunas_extras)

    dados2 = dados2.drop(columns=colunas_extras)

In [ ]:
# Colunas Faltantes

colunas_faltando = list(set_dados - set_dados2)

if colunas_faltando:
    print("Colunas adicionadas em dados2:")
    print(colunas_faltando)

    for col in colunas_faltando:
        dados2[col] = -100

In [ ]:
# Garante mesma ordem do treino

dados2 = dados2[cols_dados]

print("\nValidação concluída: colunas ajustadas com sucesso.")

In [ ]:
# -------- EXTRAÇÃO SEGURA --------
col_num = dados2["num"]
col_ambiente = dados2["ambiente"]

# Features exatamente iguais às do treino
X_novo = dados2.iloc[:, 2:]

In [ ]:
# =========================
# 5. PREDIÇÃO (DATAFRAME INTEIRO)
# =========================

# Faz a predição de todas as linhas de uma vez
pred = rf_model.predict(X_novo)

# Converte os códigos numéricos para nomes originais
predicoes = le.inverse_transform(pred)

In [ ]:
# =========================
# 4. GERAR SAÍDA
# =========================

# Data e hora atuais
agora = datetime.now()
data_str = agora.strftime("%d/%m/%Y")
hora_str = agora.strftime("%H:%M:%S")

saida = pd.DataFrame({
    "data": data_str,
    "hora": hora_str,
    "num": col_num.values,
    "local": col_ambiente.values,
    "predição": predicoes
})

In [ ]:
# =========================
# 5. SALVAR CSV FINAL
# =========================

arquivo_saida = "predicaoRF10CampoVirtual080526EE.csv"
saida.to_csv(arquivo_saida, index=False)

print(f"\nArquivo salvo em: {arquivo_saida}")